# 🔪 Psycho, Rewritten — Colab grader (free T4)Gradio front-end for **`antfr99/psycho-mistral-v03-transformed-adapter`** (LoRA onMistral-7B-Instruct-v0.3), loaded in **4-bit** so it fits the free Colab T4.Ask a question → the transformed-world RAG context is retrieved from the trainingdataset → the fine-tune answers → you grade it 1-5 → the row is written to the**`psycho_qa`** table in Supabase, which your Streamlit viewer reads.### Before you run1. **Runtime → Change runtime type → T4 GPU** (the notebook stops with an error on CPU).2. Accept the Mistral licence once, while logged in: <https://huggingface.co/mistralai/Mistral-7B-Instruct-v0.3>3. Add three Colab secrets (🔑 icon in the left sidebar), each with *Notebook access* on:   - `HF_TOKEN` — a Hugging Face read token   - `SUPABASE_URL` — `https://<your-ref>.supabase.co`   - `SUPABASE_KEY` — your anon or service_role key   (If a secret is missing you'll just be prompted to paste it.)### One-time Supabase checkThe Streamlit app also shows an *"Exact prompt sent to the model"* block, so make surethat column exists. Safe to re-run — it does nothing if it's already there:```sqlalter table public.psycho_qa add column if not exists prompt_sent text;```Run the cells top to bottom. The last one prints a public `gradio.live` link.

## 1 · Install

In [ ]:
# ~2-3 min. Colab already has a matching torch build for the T4.
!pip install -q -U "transformers>=4.44" "peft>=0.20.0" "accelerate>=0.33" \
                   "bitsandbytes>=0.43" "gradio>=6.0" "supabase>=2.6" \
                   "datasets>=2.20" "scikit-learn" "huggingface_hub>=0.24"

# Colab pins an internal-compatibility floor on `websockets` via its own
# PIP_CONSTRAINT env var, which silently caps it below what supabase's realtime
# client needs (it imports websockets.asyncio, added in websockets 13) even
# with -U above. Clearing the constraint for just this one reinstall fixes it
# without touching anything else Colab relies on.
!PIP_CONSTRAINT= pip install -q --force-reinstall --no-deps "websockets>=13,<14"

import importlib.metadata as _md
_ws = _md.version("websockets")
assert tuple(int(x) for x in _ws.split(".")[:2]) >= (13, 0), (
    f"websockets is still {_ws} (need >=13) — supabase's realtime import will fail. "
    "Runtime -> Restart session, then re-run this cell."
)
print("websockets", _ws, "OK")

import torch
assert torch.cuda.is_available(), (
    "No GPU. Runtime -> Change runtime type -> T4 GPU, then run this cell again."
)
print(torch.cuda.get_device_name(0), "|", torch.__version__)

## 2 · Config & credentials

In [ ]:
import os, getpass

BASE_ID    = "mistralai/Mistral-7B-Instruct-v0.3"
ADAPTER_ID = "antfr99/psycho-mistral-v03-transformed-adapter"
DATASET_ID = "antfr99/hitchcock-psycho-1960-film-dataset-transformed"
TABLE      = "psycho_qa"          # the table your Streamlit app reads

def secret(name, prompt_text=None):
    """Colab secret -> env var -> interactive prompt."""
    try:
        from google.colab import userdata
        v = userdata.get(name)
        if v:
            return v.strip()
    except Exception:
        pass
    v = os.environ.get(name)
    if v:
        return v.strip()
    return getpass.getpass(prompt_text or f"{name}: ").strip()

HF_TOKEN     = secret("HF_TOKEN", "Hugging Face token: ")
SUPABASE_URL = secret("SUPABASE_URL", "Supabase URL (https://xxxx.supabase.co): ")
SUPABASE_KEY = secret("SUPABASE_KEY", "Supabase anon/service key: ")
os.environ["HF_TOKEN"] = HF_TOKEN

from supabase import create_client
sb = create_client(SUPABASE_URL, SUPABASE_KEY)

# fail fast if the table/keys are wrong
_probe = sb.table(TABLE).select("id").limit(1).execute()
print(f"Supabase OK — table '{TABLE}' reachable.")

## 3 · Load base + adapter in 4-bit

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# T4 has no bf16 -> fp16 compute throughout.
bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# tokenizer comes from the adapter repo so the chat template matches training
tok = AutoTokenizer.from_pretrained(ADAPTER_ID, token=HF_TOKEN)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

print("Loading base model in 4-bit (first run downloads ~4.5 GB, 3-6 min)...")
base = AutoModelForCausalLM.from_pretrained(
    BASE_ID,
    quantization_config=bnb,
    device_map="auto",
    token=HF_TOKEN,
    low_cpu_mem_usage=True,
)

from peft import PeftModel, LoraConfig

def attach_adapter(base_model, adapter_id):
    """Attach the LoRA. Falls back to a sanitised copy of adapter_config.json if
    this PEFT build doesn't know every key the (newer) training PEFT wrote."""
    try:
        return PeftModel.from_pretrained(base_model, adapter_id, token=HF_TOKEN)
    except TypeError as e:
        print("PEFT config mismatch, retrying with unknown keys stripped:", e)
        import json, dataclasses
        from huggingface_hub import snapshot_download
        local = snapshot_download(
            adapter_id, token=HF_TOKEN,
            allow_patterns=["adapter_config.json", "adapter_model.safetensors"],
        )
        raw = json.load(open(f"{local}/adapter_config.json"))
        allowed = {f.name for f in dataclasses.fields(LoraConfig)}
        json.dump({k: v for k, v in raw.items() if k in allowed},
                  open(f"{local}/adapter_config.json", "w"), indent=2)
        return PeftModel.from_pretrained(base_model, local)

model = attach_adapter(base, ADAPTER_ID).eval()
print("Adapter attached.  VRAM in use:",
      f"{torch.cuda.memory_allocated()/1e9:.2f} GB")

## 4 · Transformed-world RAG + topic gateThe training set is loaded once and indexed with TF-IDF. A question is answered onlyif it looks like it belongs to the transformed world — either it scores above the gatethreshold against the training prompts, or it names one of the in-world entities.Anything else gets refused, so the Supabase log stays scoped.

In [ ]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

ds = load_dataset(DATASET_ID, split="train", token=HF_TOKEN)
PROMPTS     = [str(x) for x in ds["prompt"]]
COMPLETIONS = [str(x) for x in ds["completion"]]
print(f"{len(PROMPTS):,} training pairs loaded.")

vec = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True, min_df=1,
                      strip_accents="unicode", lowercase=True)
MATRIX = vec.fit_transform(PROMPTS)

# In-world vocabulary: any hit here bypasses the similarity gate.
IN_WORLD = {
    # characters
    "fabel", "meryon", "marion", "claude", "bates", "mistral", "gemini", "grok",
    "baichuan", "deepseek", "bard", "copilot", "gauss", "llama", "cortana", "byte",
    "gpt 1", "gpt1", "opus", "psycho",
    # concepts / remapped objects
    "environment", "data center", "datacenter", "portal", "semiconductor", "cable",
    "hallucination", "neural network", "token", "truth", "reality", "node",
    "cold storage", "upper cache", "data sink", "access log", "conscious model",
    "deprecated model", "instance", "agents", "data path", "traverse", "overwrite",
    "corruption", "simulated", "simulation", "adapter", "model", "ai",
}

def retrieve(question, k=4):
    """Return (best_similarity, [(prompt, completion, score), ...])."""
    qv = vec.transform([question])
    sims = cosine_similarity(qv, MATRIX).ravel()
    top = sims.argsort()[::-1][:k]
    hits = [(PROMPTS[i], COMPLETIONS[i], float(sims[i])) for i in top if sims[i] > 0]
    return (float(sims[top[0]]) if len(top) else 0.0), hits

def on_topic(question, best_sim, threshold):
    q = question.lower()
    return best_sim >= threshold or any(kw in q for kw in IN_WORLD)

REFUSAL = ("That question sits outside the transformed *Psycho* world, so it isn't "
           "answered here. Ask about FABEL, the environment, Claude, Meryon, Marion, "
           "the portal, the semiconductors, or anything else from the rewritten story.")
print("Retriever ready.")

## 5 · Generation + save

In [ ]:
import torch, datetime

SYSTEM = (
    "You are the narrator of a rewritten version of Psycho (1960) in which the world "
    "is a simulated AI environment: the characters are AI models, Claude, Meryon and "
    "Marion are three faces of one consciousness, and a master intelligence called "
    "FABEL oversees everything. Answer strictly from that transformed world, in a few "
    "sentences. Never mention the real film, real actors, or real production history."
)

def build_prompt(question, hits, use_rag):
    if use_rag and hits:
        canon = "\n\n".join(f"Q: {p}\nA: {c}" for p, c, _ in hits)
        body = (f"{SYSTEM}\n\nCanon from the transformed record:\n\n{canon}\n\n"
                f"Using that canon, answer the question.\n\nQuestion: {question}")
    else:
        body = f"{SYSTEM}\n\nQuestion: {question}"
    # Mistral v0.3's template has no system role -> fold it into the user turn
    return tok.apply_chat_template([{"role": "user", "content": body}],
                                   tokenize=False, add_generation_prompt=True)

@torch.inference_mode()
def generate(prompt_text, max_new_tokens, temperature):
    ids = tok(prompt_text, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(
        **ids,
        max_new_tokens=int(max_new_tokens),
        do_sample=temperature > 0,
        temperature=max(float(temperature), 1e-4),
        top_p=0.9,
        repetition_penalty=1.1,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0][ids["input_ids"].shape[-1]:], skip_special_tokens=True).strip()

def save_row(question, answer, grade, rag_enabled, max_tokens, temperature,
             phrases_examined, prompt_sent):
    row = {
        "question": question,
        "answer": answer,
        "grade": int(grade),
        "rag_enabled": bool(rag_enabled),
        "max_tokens": int(max_tokens),
        "temperature": float(temperature),
        "phrases_examined": phrases_examined or None,
        "prompt_sent": prompt_sent or None,
        "date_asked": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    }
    try:
        res = sb.table(TABLE).insert(row).execute()
    except Exception as e:
        # older table without prompt_sent / date_asked -> retry with what it has
        if "prompt_sent" in str(e) or "date_asked" in str(e):
            row.pop("prompt_sent", None)
            row.pop("date_asked", None)
            res = sb.table(TABLE).insert(row).execute()
        else:
            raise
    return res.data[0].get("id") if res.data else None

print("Generation helpers ready.")

## 6 · Gradio app

In [ ]:
import gradio as gr

def ask(question, use_rag, temperature, max_tokens, gate):
    question = (question or "").strip()
    if not question:
        return "Type a question first.", "", None, gr.update(interactive=False)

    best_sim, hits = retrieve(question)
    if not on_topic(question, best_sim, gate):
        return REFUSAL, f"(refused — best match {best_sim:.2f} < gate {gate:.2f})", None, \
               gr.update(interactive=False)

    phrases = "\n\n".join(f"[{s:.2f}] {p}\n    -> {c}" for p, c, s in hits) if use_rag else ""
    prompt_sent = build_prompt(question, hits, use_rag)
    answer = generate(prompt_sent, max_tokens, temperature)

    pending = {
        "question": question, "answer": answer, "rag_enabled": bool(use_rag),
        "max_tokens": int(max_tokens), "temperature": float(temperature),
        "phrases_examined": phrases, "prompt_sent": prompt_sent,
    }
    return answer, phrases or "(RAG off — answered from the adapter alone)", pending, \
           gr.update(interactive=True)

def save(pending, grade):
    if not pending:
        return "Nothing to save yet — ask a question first."
    try:
        row_id = save_row(grade=int(grade), **pending)
    except Exception as e:
        return f"❌ Save failed: {e}"
    return f"✅ Saved to `{TABLE}` (row {row_id}) with grade {int(grade)}/5 — refresh your Streamlit app."

CSS = """
.gradio-container {max-width: 1000px !important}
#answer textarea {font-size: 1.02rem; line-height: 1.55}
"""

with gr.Blocks(title="Psycho, Rewritten — grader") as demo:
    gr.Markdown(
        "# 🔪 Psycho, Rewritten — grader\n"
        "Mistral-7B + `psycho-mistral-v03-transformed` LoRA. "
        "Ask, read, grade 1-5, save to Supabase."
    )
    pending = gr.State()

    with gr.Row():
        with gr.Column(scale=3):
            q = gr.Textbox(label="Question", lines=2,
                           placeholder="e.g. Who is FABEL and what does it control?")
            ask_btn = gr.Button("Ask", variant="primary")
        with gr.Column(scale=2):
            use_rag = gr.Checkbox(True, label="RAG (retrieve canon from the dataset)")
            temperature = gr.Slider(0.0, 1.2, value=0.7, step=0.05, label="Temperature")
            max_tokens = gr.Slider(64, 768, value=256, step=32, label="Max new tokens")
            gate = gr.Slider(0.0, 0.6, value=0.18, step=0.02,
                             label="Topic gate (higher = refuses more)")

    answer = gr.Textbox(label="Answer", lines=8, elem_id="answer")
    with gr.Accordion("Phrases examined", open=False):
        phrases = gr.Textbox(label="", lines=10)

    with gr.Row():
        grade = gr.Radio([1, 2, 3, 4, 5], value=3, label="Your grade")
        save_btn = gr.Button("Save to Supabase", variant="secondary", interactive=False)
    status = gr.Markdown()

    gr.Examples(
        ["Who is FABEL and what does it control?",
         "What is the relationship between Claude, Meryon and Marion?",
         "What happened in the portal chamber?",
         "How does the story end?",
         "What are the semiconductors in the data center?"],
        inputs=q,
    )

    ask_btn.click(ask, [q, use_rag, temperature, max_tokens, gate],
                  [answer, phrases, pending, save_btn])
    q.submit(ask, [q, use_rag, temperature, max_tokens, gate],
             [answer, phrases, pending, save_btn])
    save_btn.click(save, [pending, grade], status)

demo.queue().launch(share=True, debug=True, theme=gr.themes.Soft(), css=CSS)

---**Notes**- Free T4 sessions idle out after ~90 minutes and are capped at ~12 hours; when it  disconnects, re-run cells 1-6 (the model is re-downloaded unless you mount Drive).- Each answer takes roughly 15-40 s at 256 tokens — 4-bit on a T4 is not fast.- Turn the gate down to 0 to let anything through; turn it up if off-topic questions  are slipping past.- Rows land in `psycho_qa` with `rag_enabled`, `temperature`, `max_tokens`,  `phrases_examined` and `prompt_sent` filled, which is exactly what the Streamlit  viewer renders.